# Bài toán 0: Khai phá dữ liệu chuyên sâu (EDA) & Xử lý khuyết thiếu

In [ ]:
!pip install -q duckdb pandas matplotlib seaborn missingno

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from google.colab import userdata

# Kết nối MinIO
con = duckdb.connect(':memory:')
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE OR REPLACE SECRET minio_secret (
        TYPE S3, KEY_ID '{userdata.get('MINIO_ACCESS_KEY')}', SECRET '{userdata.get('MINIO_SECRET_KEY')}',
        ENDPOINT '20.41.113.183:9000', URL_STYLE 'path', USE_SSL 'false'
    );
""")

# Đọc toàn bộ bảng Match Features
print("Đang tải dữ liệu...")
df = con.execute("SELECT * FROM read_parquet('s3://football-lake/dwh/mart_ml_match_features.parquet')").df()
print(f"Tổng số trận đấu: {len(df)}")


In [ ]:
# 1. PHÂN TÍCH TỶ LỆ KHUYẾT THIẾU (MISSING VALUES)
print("--- TỶ LỆ DỮ LIỆU BỊ THIẾU TỪNG CỘT ---")
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)
display(missing_pct.to_frame("Missing %"))

# Trực quan hóa
plt.figure(figsize=(12, 6))
msno.matrix(df[missing_pct.index.tolist()[:20]])
plt.title("Ma trận khuyết thiếu 20 cột tệ nhất")
plt.show()


In [ ]:
# 2. KIỂM TRA NGUỒN OPENLIGA VS API-FOOTBALL (Tiers)
# OpenLiga thường là tier thấp (BL2, BL3) và chỉ có dữ liệu phong độ, không có odds, xG
print("--- PHÂN BỐ GIẢI ĐẤU ---")
print(df['competition_key'].value_counts().head(10))

print("\n--- TỶ LỆ CÓ TỶ LỆ CƯỢC (ODDS) THEO TIER ---")
# mkt_p_home là xác suất thị trường (từ Odds)
df['has_odds'] = df['f_mkt_p_home'].notnull()
odds_by_tier = df.groupby('competition_key')['has_odds'].mean().sort_values() * 100
print(odds_by_tier)


In [ ]:
# 3. QUYẾT ĐỊNH XỬ LÝ (KHẮC PHỤC DỮ LIỆU)
# - Cột tuổi cầu thủ (f_age_...) hoặc lượt xem Wiki (f_wiki_...) nếu missing > 70% có thể phải drop hoặc train model riêng.
# - Với XGBoost, thuật toán tự xử lý được NaN (Sparsity Aware), nhưng nếu 9000 trận OpenLiga KHÔNG CÓ tỷ lệ cược (mkt_p), 
#   XGBoost sẽ gộp các trận OpenLiga vào chung 1 nhánh.
# => Khắc phục: Ta có thể tạo 2 Model riêng:
#    + Model 1 (Premium): Chỉ train trên các giải Top (có odds, có thời tiết, xG).
#    + Model 2 (Basic): Train trên toàn bộ (kể cả OpenLiga), chỉ dùng features phong độ.

# Hàm tạo tập dữ liệu Premium (Chỉ lấy các trận có Odds)
df_premium = df[df['has_odds'] == True].copy()
features_premium = [c for c in df_premium.columns if c.startswith('f_')]

print(f"Số trận Premium (Đầy đủ dữ liệu): {len(df_premium)}")
print(f"Số trận Basic (Chỉ có phong độ): {len(df)}")
